# UMUD — Apo Contrast Context Fill (Phase 3)

**GPU notebook** — tests your “contrast/context” hypothesis.

We take each test ultrasound image and:
1. Compute the ultrasound ROI bbox (non-black content).
2. Replace everything outside that bbox with a neutral **gray** value.
3. Run apo segmentation + Phase-2 geometry on both:
   - **Baseline**: raw image
   - **Gray-fill**: gray-context image

Outputs:
- `/kaggle/working/apo_contrast_fill_compare.csv`

The notebook also displays a small gallery of cases where you can visually inspect:
Raw image -> Gray-filled image -> Predicted mask(s) + overlays + inverted masks.



## Configuration


In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import matplotlib.pyplot as plt

IMG_SIZE = 256
APO_REGION_THRESHOLD = 0.50

N_GALLERY_FAIL = 10
N_GALLERY_OK = 6
RANDOM_SEED = 42

ROI_THRESH = 5        # non-black threshold for bbox
ROI_PAD_PX = 10       # bbox padding

MASK_OVERLAY_ALPHA = 0.55
APO_OVERLAY_COLOR = (255, 140, 0)

# How to compute the neutral gray fill value.
# We take a dark-pixel percentile from the image, but clamp to a minimum.
GRAY_DARK_PERCENTILE = 10  # lower percentile => darker gray
GRAY_MIN_VALUE = 15        # avoid turning gutters into near-black

# Optional contrast stretch inside ROI (OFF by default; enabled only if you want).
DO_CONTRAST_STRETCH = False
P_LOW = 1
P_HIGH = 99

FIG_DIR = Path("/kaggle/working/figures/apo_contrast_fill")
FIG_DIR.mkdir(parents=True, exist_ok=True)

APO_MODEL_PATH = Path(
    "/kaggle/input/notebooks/ucheozoemena/umud-train-apo-mounted-phase-3/apo_baseline.pkl"
)

COMPETITION_DIR = Path(
    "/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data"
)
TEST_DIR = COMPETITION_DIR / "test_images_v2/test_set_v2"


In [ ]:
from __future__ import annotations

import cv2
import numpy as np
import pandas as pd
from fastai.vision.all import PILImage, load_learner
from PIL import Image
from tqdm.auto import tqdm

IMAGE_EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}


def list_test_images(directory: Path) -> list[Path]:
    files = [
        p
        for p in directory.rglob("*")
        if p.suffix.lower() in IMAGE_EXTS and p.name != "Thumbs.db"
    ]
    # Prefer .tif when both .tif and .png exist for same stem
    by_stem: dict[str, Path] = {}
    for p in sorted(files):
        stem = p.stem
        if stem not in by_stem or p.suffix.lower() in {".tif", ".tiff"}:
            by_stem[stem] = p
    return sorted(by_stem.values(), key=lambda p: p.name)


def load_gray(path: Path) -> np.ndarray:
    with Image.open(path) as im:
        arr = np.array(im)
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    return arr.astype(np.uint8)


def resize_image(img: np.ndarray, size: int) -> np.ndarray:
    return np.array(Image.fromarray(img).resize((size, size), Image.BILINEAR), dtype=np.uint8)


def resize_mask_to(mask: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    if mask.shape == (target_h, target_w):
        return (mask > 0).astype(np.uint8)
    src = (mask > 0).astype(np.uint8) * 255
    out = Image.fromarray(src).resize((target_w, target_h), Image.NEAREST)
    return (np.array(out) > 0).astype(np.uint8)


def tensor_to_mask(pred) -> np.ndarray:
    if hasattr(pred, "cpu"):
        pred = pred.cpu().numpy()
    arr = np.asarray(pred)
    if arr.ndim == 3:
        arr = arr.argmax(axis=0)
    return (arr > 0).astype(np.uint8)


def open_rgb_256(img_native: np.ndarray) -> PILImage:
    small = resize_image(img_native, IMG_SIZE)
    rgb = np.stack([small, small, small], axis=-1).astype(np.uint8)
    return PILImage.create(rgb)


def tag_apo_style(coverage: float) -> str:
    return "region" if coverage >= APO_REGION_THRESHOLD else "line"


def invert_mask(mask: np.ndarray) -> np.ndarray:
    return (1 - mask).astype(np.uint8)


def effective_apo_mask(mask: np.ndarray, style: str) -> tuple[np.ndarray, str]:
    if style == "region":
        return invert_mask(mask), "inverted_region"
    return mask, "raw_line"


def find_apo_contours(mask: np.ndarray, min_area_frac: float = 0.0003) -> list[np.ndarray]:
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    min_area = mask.size * min_area_frac
    big = [c for c in contours if cv2.contourArea(c) >= min_area]
    big.sort(key=lambda c: cv2.boundingRect(c)[1])
    return big


def pick_superficial_deep(contours: list[np.ndarray], min_sep_px: int = 15):
    if len(contours) < 2:
        return None, None, len(contours)
    sup = contours[0]
    _, y0, _, _ = cv2.boundingRect(sup)
    deep = None
    for c in contours[1:]:
        _, y1, _, _ = cv2.boundingRect(c)
        if y1 >= y0 + min_sep_px:
            deep = c
            break
    if deep is None:
        deep = contours[min(2, len(contours) - 1)]
    return sup, deep, len(contours)


def edge_polyline(contour: np.ndarray, which: str = "bottom", n_bins: int = 60):
    pts = contour.reshape(-1, 2)
    if len(pts) < 3:
        return np.array([]), np.array([])
    x_min, x_max = pts[:, 0].min(), pts[:, 0].max()
    if x_max <= x_min:
        return pts[:, 0].astype(float), pts[:, 1].astype(float)
    edges = np.linspace(x_min, x_max, n_bins + 1)
    xs_out, ys_out = [], []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        in_bin = pts[(pts[:, 0] >= lo) & (pts[:, 0] < hi)]
        if len(in_bin) == 0:
            continue
        y = in_bin[:, 1].max() if which == "bottom" else in_bin[:, 1].min()
        xs_out.append((lo + hi) / 2.0)
        ys_out.append(float(y))
    return np.array(xs_out), np.array(ys_out)


def fit_line(xs: np.ndarray, ys: np.ndarray):
    if len(xs) < 2:
        return None
    return np.poly1d(np.polyfit(xs, ys, 1))


def line_angle_deg(line) -> float:
    return float(np.degrees(np.arctan(line[1]))) if line is not None else np.nan


def mt_from_apo_edges(sup_line, deep_line, x_left: float, x_right: float) -> float:
    if sup_line is None or deep_line is None or x_right <= x_left:
        return np.nan
    thirds = [x_left + (x_right - x_left) * t for t in (1 / 6, 3 / 6, 5 / 6)]
    dists = [abs(float(deep_line(x) - sup_line(x))) for x in thirds]
    return float(np.mean(dists))


def fascicle_pca(mask: np.ndarray) -> dict | None:
    ys, xs = np.where(mask > 0)
    if len(xs) < 3:
        return None
    coords = np.column_stack([xs.astype(float), ys.astype(float)])
    centered = coords - coords.mean(axis=0)
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    direction = vh[0]
    projections = centered @ direction
    return {
        "length_px": float(projections.max() - projections.min()),
        "angle_deg": float(np.degrees(np.arctan2(direction[1], direction[0]))),
    }


def acute_angle_deg(a1: float, a2: float) -> float:
    d = abs(a1 - a2) % 180.0
    return float(min(d, 180.0 - d))


def apo_geometry_attempt(apo_mask: np.ndarray, style: str) -> dict:
    eff, method = effective_apo_mask(apo_mask, style)
    contours = find_apo_contours(eff)
    sup_c, deep_c, n_contours = pick_superficial_deep(contours)
    out = {
        "apo_method": method,
        "n_contours": n_contours,
        "mt_px": np.nan,
        "deep_angle_deg": np.nan,
        "mt_fail_reason": "ok",
        "sup_line": None,
        "deep_line": None,
        "sup_xs": None,
        "sup_ys": None,
        "deep_xs": None,
        "deep_ys": None,
    }
    if len(contours) == 0:
        out["mt_fail_reason"] = "no_contours"
        return out
    if n_contours < 2 or sup_c is None or deep_c is None:
        out["mt_fail_reason"] = "single_contour"
        return out
    sup_x, sup_y = edge_polyline(sup_c, which="bottom")
    deep_x, deep_y = edge_polyline(deep_c, which="top")
    sup_line = fit_line(sup_x, sup_y)
    deep_line = fit_line(deep_x, deep_y)
    out.update(sup_line=sup_line, deep_line=deep_line, sup_xs=sup_x, sup_ys=sup_y, deep_xs=deep_x, deep_ys=deep_y)
    if sup_line is None or deep_line is None:
        out["mt_fail_reason"] = "line_fit_fail"
        return out
    if len(sup_x) == 0 or len(deep_x) == 0:
        out["mt_fail_reason"] = "empty_edge_polyline"
        return out
    x_left = max(sup_x.min(), deep_x.min())
    x_right = min(sup_x.max(), deep_x.max())
    if x_right <= x_left:
        out["mt_fail_reason"] = "no_x_overlap"
        return out
    out["mt_px"] = mt_from_apo_edges(sup_line, deep_line, x_left, x_right)
    out["deep_angle_deg"] = line_angle_deg(deep_line)
    if np.isnan(out["mt_px"]):
        out["mt_fail_reason"] = "mt_compute_nan"
    return out


def apo_geometry_from_mask(apo_mask: np.ndarray, style: str) -> dict:
    primary = apo_geometry_attempt(apo_mask, style)
    primary["geometry_path"] = primary["apo_method"]
    if not np.isnan(primary["mt_px"]):
        return primary
    if style == "region":
        fallback = apo_geometry_attempt(apo_mask, "line")
        if not np.isnan(fallback["mt_px"]):
            fallback["apo_method"] = f"{primary['apo_method']}+fallback_line"
            fallback["geometry_path"] = "fallback_line"
            fallback["mt_fail_reason_primary"] = primary["mt_fail_reason"]
            return fallback
    primary["mt_fail_reason_primary"] = primary["mt_fail_reason"]
    return primary


def derive_geometry(fasc_mask: np.ndarray, apo_mask: np.ndarray, apo_style: str) -> dict:
    apo = apo_geometry_from_mask(apo_mask, apo_style)
    fpca = fascicle_pca(fasc_mask)
    out = {
        "pa_deg": np.nan,
        "fl_px": np.nan,
        "mt_px": apo["mt_px"],
        "apo_method": apo.get("apo_method"),
        "geometry_path": apo.get("geometry_path"),
        "n_contours": apo.get("n_contours"),
        "mt_fail_reason": apo.get("mt_fail_reason"),
        "mt_fail_reason_primary": apo.get("mt_fail_reason_primary"),
        "apo_cov": float(apo_mask.mean()),
        "apo_fg_pixels": int(apo_mask.sum()),
        "fasc_cov": float(fasc_mask.mean()),
        "fasc_fg_pixels": int(fasc_mask.sum()),
    }
    if fpca is not None:
        out["fl_px"] = fpca["length_px"]
        ref = apo["deep_angle_deg"] if not np.isnan(apo["deep_angle_deg"]) else 0.0
        out["pa_deg"] = acute_angle_deg(fpca["angle_deg"], ref)
    return out


In [ ]:
def overlay(img_gray: np.ndarray, mask: np.ndarray, color=APO_OVERLAY_COLOR, alpha=MASK_OVERLAY_ALPHA):
    rgb = np.stack([img_gray, img_gray, img_gray], axis=-1).astype(np.float32)
    tint = np.zeros_like(rgb)
    tint[..., 0], tint[..., 1], tint[..., 2] = color
    sel = mask.astype(bool)
    rgb[sel] = (1 - alpha) * rgb[sel] + alpha * tint[sel]
    return rgb.astype(np.uint8)


def find_roi_bbox(img_gray: np.ndarray, thr: float = ROI_THRESH, pad: int = ROI_PAD_PX):
    """Ultrasound ROI bbox from non-black pixels.

    Uses connected components to suppress small speckle blobs.
    Returns (y0, y1, x0, x1) in native coordinates.
    """
    import cv2

    roi = (img_gray > thr).astype(np.uint8)
    if roi.sum() == 0:
        h, w = img_gray.shape
        return 0, h, 0, w

    num, labels, stats, _ = cv2.connectedComponentsWithStats(roi, connectivity=8)
    best = max(range(1, num), key=lambda i: stats[i, cv2.CC_STAT_AREA])
    x, y, w, h = stats[best, :4]

    y0 = max(0, y - pad)
    y1 = min(img_gray.shape[0], y + h + pad)
    x0 = max(0, x - pad)
    x1 = min(img_gray.shape[1], x + w + pad)
    return int(y0), int(y1), int(x0), int(x1)


def compute_gray_fill_value(img_gray: np.ndarray):
    """Pick a dark-but-not-black gray from the image background."""
    flat = img_gray.reshape(-1)
    # Ignore exact zeros when possible (pure black borders).
    nonzero = flat[flat > 0]
    if len(nonzero) > 1000:
        dark_base = np.percentile(nonzero, GRAY_DARK_PERCENTILE)
    else:
        dark_base = np.percentile(flat, GRAY_DARK_PERCENTILE)
    return float(max(GRAY_MIN_VALUE, dark_base))


def maybe_contrast_stretch(img_gray: np.ndarray, y0: int, y1: int, x0: int, x1: int):
    if not DO_CONTRAST_STRETCH:
        return img_gray

    out = img_gray.astype(np.float32).copy()
    roi = out[y0:y1, x0:x1]
    lo = np.percentile(roi, P_LOW)
    hi = np.percentile(roi, P_HIGH)
    if hi <= lo + 1e-6:
        return img_gray

    # Linear rescale ROI only to [0,255], clipping to remove outliers.
    roi_clipped = np.clip(roi, lo, hi)
    roi_stretched = (roi_clipped - lo) / (hi - lo) * 255.0
    out[y0:y1, x0:x1] = roi_stretched
    return out.astype(np.uint8)


def preprocess_gray_fill(img_native: np.ndarray):
    """Replace outside ROI bbox with a neutral gray, optionally stretch inside ROI."""
    h, w = img_native.shape
    y0, y1, x0, x1 = find_roi_bbox(img_native)
    gray_val = compute_gray_fill_value(img_native)

    pre = img_native.copy().astype(np.float32)
    pre_mask = np.ones((h, w), dtype=bool)
    pre_mask[y0:y1, x0:x1] = False  # True outside ROI
    pre[pre_mask] = gray_val
    pre = pre.astype(np.uint8)
    pre = maybe_contrast_stretch(pre, y0, y1, x0, x1)
    return pre, (y0, y1, x0, x1), gray_val


In [ ]:
assert APO_MODEL_PATH.exists(), f"Missing apo model: {APO_MODEL_PATH}"
assert TEST_DIR.exists(), f"Missing test dir: {TEST_DIR}"

apo_learn = load_learner(APO_MODEL_PATH)
test_paths = list_test_images(TEST_DIR)
print(f"Test images: {len(test_paths)}")


In [ ]:
rows = []

for path in tqdm(test_paths, desc="baseline + gray-fill infer"):
    img_native = load_gray(path)
    h, w = img_native.shape

    # Baseline
    pil = open_rgb_256(img_native)
    _, apo_t, _ = apo_learn.predict(pil)
    apo_native = resize_mask_to(tensor_to_mask(apo_t), h, w)
    base_cov = float(apo_native.mean())
    base_style = tag_apo_style(base_cov)
    base_geo = apo_geometry_from_mask(apo_native, base_style)

    # Gray-fill preprocessing
    img_pre, bbox, gray_val = preprocess_gray_fill(img_native)
    pil_pre = open_rgb_256(img_pre)
    _, apo_t2, _ = apo_learn.predict(pil_pre)
    apo_pre_native = resize_mask_to(tensor_to_mask(apo_t2), h, w)
    pre_cov = float(apo_pre_native.mean())
    pre_style = tag_apo_style(pre_cov)
    pre_geo = apo_geometry_from_mask(apo_pre_native, pre_style)

    rows.append(
        {
            "image_id": path.name,
            "res": f"{h}x{w}",
            "bbox": f"{bbox[0]}:{bbox[1]}:{bbox[2]}:{bbox[3]}",
            "gray_val": float(gray_val),
            "base_pred_cov": base_cov,
            "pre_pred_cov": pre_cov,
            "base_style": base_style,
            "pre_style": pre_style,
            "base_mt_ok": bool(not np.isnan(base_geo["mt_px"])),
            "pre_mt_ok": bool(not np.isnan(pre_geo["mt_px"])),
            "base_mt_fail_reason": base_geo["mt_fail_reason"],
            "pre_mt_fail_reason": pre_geo["mt_fail_reason"],
        }
    )

df = pd.DataFrame(rows)
out_csv = "/kaggle/working/apo_contrast_fill_compare.csv"
df.to_csv(out_csv, index=False)
print("Wrote:", out_csv)

print()
print("=== Overall ===")
print("base mt_ok rate:", float(df.base_mt_ok.mean()))
print("pre  mt_ok rate:", float(df.pre_mt_ok.mean()))

print()
print("=== base fail counts ===")
print(df.loc[~df.base_mt_ok, "base_mt_fail_reason"].value_counts().to_dict())

print()
print("=== pre fail counts ===")
print(df.loc[~df.pre_mt_ok, "pre_mt_fail_reason"].value_counts().to_dict())

fixed = df[(~df.base_mt_ok) & (df.pre_mt_ok)]
print()
print("MT-fixed count (baseline NaN -> gray finite):", len(fixed))


In [ ]:
# Gallery selection
fail_cases = df[df.base_mt_fail_reason == "no_contours"].copy()
ok_cases = df[df.base_mt_ok].copy()

print("no_contours baseline count:", len(fail_cases))
print("MT OK baseline count:", len(ok_cases))

rng = random.Random(RANDOM_SEED)
fail_pick = fail_cases.sample(min(N_GALLERY_FAIL, len(fail_cases)), random_state=RANDOM_SEED) if len(fail_cases) else fail_cases
ok_pick = ok_cases.sample(min(N_GALLERY_OK, len(ok_cases)), random_state=RANDOM_SEED) if len(ok_cases) else ok_cases

def show_one(image_id: str):
    path = TEST_DIR / image_id
    img_native = load_gray(path)
    h, w = img_native.shape

    # Baseline pred
    pil = open_rgb_256(img_native)
    _, apo_t, _ = apo_learn.predict(pil)
    apo_native = resize_mask_to(tensor_to_mask(apo_t), h, w)
    base_cov = float(apo_native.mean())
    base_style = tag_apo_style(base_cov)
    base_geo = apo_geometry_from_mask(apo_native, base_style)

    # Preprocess pred
    img_pre, bbox, gray_val = preprocess_gray_fill(img_native)
    pil_pre = open_rgb_256(img_pre)
    _, apo_t2, _ = apo_learn.predict(pil_pre)
    apo_pre_native = resize_mask_to(tensor_to_mask(apo_t2), h, w)
    pre_cov = float(apo_pre_native.mean())
    pre_style = tag_apo_style(pre_cov)
    pre_geo = apo_geometry_from_mask(apo_pre_native, pre_style)

    inv_base = invert_mask(apo_native)
    inv_pre = invert_mask(apo_pre_native)

    fig, axes = plt.subplots(2, 6, figsize=(32, 8))
    axes = axes.reshape(-1)

    # Row 0: raw vs preprocessed + masks/overlays baseline
    axes[0].imshow(img_native, cmap="gray")
    axes[0].set_title("raw image", fontsize=10)
    axes[0].axis("off")

    axes[1].imshow(img_pre, cmap="gray")
    axes[1].set_title(f"gray-fill image\ngray_val={gray_val:.1f}", fontsize=10)
    axes[1].axis("off")

    axes[2].imshow(apo_native, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title(f"base pred mask\n{base_style} cov={base_cov*100:.2f}%", fontsize=10)
    axes[2].axis("off")

    axes[3].imshow(inv_base, cmap="gray", vmin=0, vmax=1)
    axes[3].set_title(f"base inverted\n{base_geo['mt_fail_reason']}", fontsize=10)
    axes[3].axis("off")

    axes[4].imshow(overlay(img_native, apo_native))
    axes[4].set_title("base overlay (pred)", fontsize=10)
    axes[4].axis("off")

    axes[5].imshow(overlay(img_native, inv_base))
    axes[5].set_title("base overlay (inv)", fontsize=10)
    axes[5].axis("off")

    # Row 1: masks/overlays after gray-fill
    axes[6].imshow(apo_pre_native, cmap="gray", vmin=0, vmax=1)
    axes[6].set_title(f"pre pred mask\n{pre_style} cov={pre_cov*100:.2f}%", fontsize=10)
    axes[6].axis("off")

    axes[7].imshow(inv_pre, cmap="gray", vmin=0, vmax=1)
    axes[7].set_title(f"pre inverted\n{pre_geo['mt_fail_reason']}", fontsize=10)
    axes[7].axis("off")

    axes[8].imshow(overlay(img_pre, apo_pre_native))
    axes[8].set_title("pre overlay (pred)", fontsize=10)
    axes[8].axis("off")

    axes[9].imshow(overlay(img_pre, inv_pre))
    axes[9].set_title("pre overlay (inv)", fontsize=10)
    axes[9].axis("off")

    # bbox diagnostic
    y0,y1,x0,x1 = bbox
    rect = plt.Rectangle((x0,y0), x1-x0, y1-y0, fill=False, edgecolor='cyan', linewidth=2)
    axes[10].imshow(img_native, cmap="gray")
    axes[10].add_patch(rect)
    axes[10].set_title("bbox on raw", fontsize=10)
    axes[10].axis("off")

    axes[11].imshow(img_pre, cmap="gray")
    axes[11].add_patch(rect)
    axes[11].set_title("bbox on gray-fill", fontsize=10)
    axes[11].axis("off")

    plt.suptitle(
        f"{image_id} | base mt_ok={base_geo['mt_px'] if not np.isnan(base_geo['mt_px']) else 'NaN'} "
        f"| pre mt_ok={pre_geo['mt_px'] if not np.isnan(pre_geo['mt_px']) else 'NaN'}",
        y=0.98,
        fontsize=12,
    )
    plt.tight_layout()
    return fig


print("\n=== Gallery: baseline no_contours (fail) ===")
for i, image_id in enumerate(fail_pick.image_id.tolist(), start=1):
    print(f"[fail {i}] {image_id}")
    fig = show_one(image_id)
    fig_path = FIG_DIR / f"gallery_fail_{i:02d}_{image_id.replace('.tif','')}.png"
    fig.savefig(fig_path, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)

print("\n=== Gallery: baseline MT OK (success) ===")
for i, image_id in enumerate(ok_pick.image_id.tolist(), start=1):
    print(f"[ok {i}] {image_id}")
    fig = show_one(image_id)
    fig_path = FIG_DIR / f"gallery_ok_{i:02d}_{image_id.replace('.tif','')}.png"
    fig.savefig(fig_path, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)

print("Done. Figures in:", FIG_DIR)
